### Check toyota notebooks for regression models and SHAP explanations extraction

In [4]:
from cmath import exp
import sys
from threading import local
import pandas as pd
import numpy as np
import shap
from xgboost.sklearn import XGBClassifier
from sklearn.metrics import r2_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import sys 
import os

TRAIN MODEL

In [77]:
def calc_shap(model, x_explain):
    explainer = shap.KernelExplainer(model.predict_proba, X_train)        
    shap_values = explainer.shap_values(x_explain)
    return shap_values, explainer

from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = MLPClassifier(solver='lbfgs', alpha=1e-1, hidden_layer_sizes=(5, 2), random_state=0)
model.fit(X_train, y_train)

MLPClassifier(alpha=0.1, hidden_layer_sizes=(5, 2), random_state=0,
              solver='lbfgs')

CALCULATE SHAP VALUES

In [82]:
# get the shapley values for the whole dataset
shap_values, explainer = calc_shap(model, X_test)

Using 426 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.
  0%|          | 0/143 [00:00<?, ?it/s]The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LassoLarsIC())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)

Set parameter alpha to: original_alpha * np.sqrt(n_samples). 
The default of 'normalize' will be set to False in version 1.2 and deprecated in version 1.4.
If you wish to scale the data, use Pipeline with a StandardScaler in a pre

In [83]:
shap_values = shap_values[0] + shap_values[1]

EXTRACT SHAP VALUES

In [100]:
# ----------- Extract global explanations -----------
feature_names = load_breast_cancer().feature_names
shap_df = pd.DataFrame(shap_values, columns=feature_names)
vals = np.abs(shap_df.values).mean(0) # get the mean of the shap values for each feature
shap_importance = pd.DataFrame(list(zip(feature_names, vals)), columns=['col_name', 'feature_importance_vals'])
shap_importance.sort_values(by=['feature_importance_vals'], ascending=False, inplace=True)

shap_importance["col_name"] =  'Feature: ' + shap_importance["col_name"].astype(str)
shap_importance = shap_importance.head()
shap_importance.to_csv(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_SHAP\features.txt', header=None, index=None, sep=' ', mode='a')


IMPLEMENT LOOP SCRIPT

--------------------------

In [137]:
experiments = 3

In [147]:
values = []
features = []
for i in range(experiments):
    for j in range(5):
        x = data[i*8+j].split()
        temp = (x[2].split('"'))
        values.append(x[3])
        features.append(x[1]+" "+str(temp[0]))


In [139]:
# Make dataframe with the shapley values of the features for each run  
values = np.array(values)
sub_arrays = np.hsplit(values, experiments)
df = pd.DataFrame(sub_arrays)
for i in range(experiments):
    df.index = ['Run_%d'%(i+1) for i in range(experiments)]
df.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']

In [140]:
# Make dataframe with the feature names (ranking of the features at each experiment)
features = np.array(features)
sub_arrays2 = np.hsplit(features, experiments)
df2 = pd.DataFrame(sub_arrays2)
for i in range(experiments):
    df2.index = ['Run_%d'%(i+1) for i in range(experiments)]
df2.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']
df2 = df2.applymap(lambda x: x.replace('"', ''))

In [141]:
# Gather here the unique features
uniqueValues = (df2['1st Feature'].append(df2['2nd Feature'].append(df2['3rd Feature'].append(df2['4th Feature'].append(df2['5th Feature']))))).unique()

df2 = df2.astype(str)